# Localbricks Databricks Basics

This notebook walks through the local training stack: environment variables, Spark, Delta Lake, Unity Catalog, PDF parsing, LangChain chunking, and Unity Catalog AI tools.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv('/workspace/.env')

UC_URI = os.getenv('UC_URI', 'http://uc-server:8080')
UC_API_URL = os.getenv('UC_API_URL', f'{UC_URI}/api/2.1/unity-catalog')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
WAREHOUSE = Path('/workspace/warehouse')
PDF_DIR = Path('/workspace/data/raw/pdfs')

WAREHOUSE.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

print(f'Unity Catalog API: {UC_API_URL}')
print(f'PDF directory: {PDF_DIR}')
print(f'OpenAI key configured: {bool(os.getenv("OPENAI_API_KEY"))}')

Unity Catalog API: http://uc-server:8080/api/2.1/unity-catalog
PDF directory: /workspace/data/raw/pdfs
OpenAI key configured: False


## Start Spark With Delta Lake And Unity Catalog

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('localbricks-training')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.1,io.unitycatalog:unitycatalog-spark_2.12:0.2.0')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'io.unitycatalog.spark.UCSingleCatalog')
    .config('spark.sql.catalog.unity', 'io.unitycatalog.spark.UCSingleCatalog')
    .config('spark.sql.catalog.unity.uri', UC_URI)
    .config('spark.sql.catalog.unity.token', '')
    .config('spark.sql.defaultCatalog', 'unity')
    .config('spark.databricks.delta.schema.autoMerge.enabled', 'true')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
spark.version

/workspace/.venv/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


:: loading settings :: url = jar:file:/workspace/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/notebook/.ivy2/cache
The jars for the packages stored in: /home/notebook/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
io.unitycatalog#unitycatalog-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1a86e264-c6d3-4a29-af30-2a6a7ccef07d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found io.unitycatalog#unitycatalog-spark_2.12;0.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.2.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.23.1 in central
	found org.apache.logging.log4j#log4j-api;2.23.1 in central
	found org.apache.logging.log4j#log4j-core;2.23.1 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jsr310;2.17.0 in central
	found org.openapitools#jackson-databind-nullable;0.2.6 in centra

'3.5.3'

In [3]:
spark.sql('SHOW SCHEMAS').show(truncate=False)
spark.sql('SHOW TABLES IN default').show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
+---------+

+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|default  |marksheet        |false      |
|default  |marksheet_uniform|false      |
|default  |numbers          |false      |
|default  |user_countries   |false      |
+---------+-----------------+-----------+



## Create And Query A Delta Table

In [4]:
import shutil

demo_table_path = WAREHOUSE / 'demo' / 'training_events'

spark.sql('CREATE SCHEMA IF NOT EXISTS demo')
spark.sql('DROP TABLE IF EXISTS demo.training_events')
shutil.rmtree(demo_table_path, ignore_errors=True)

spark.sql(f'''
CREATE TABLE demo.training_events (
  id INT,
  topic STRING,
  status STRING
)
USING delta
LOCATION '{demo_table_path.as_posix()}'
''')

spark.sql("""
INSERT INTO demo.training_events VALUES
  (1, 'spark', 'started'),
  (2, 'delta', 'started'),
  (3, 'unity_catalog', 'started')
""")

spark.sql('SELECT * FROM demo.training_events ORDER BY id').show(truncate=False)

26/04/27 13:36:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+-------------+-------+
|id |topic        |status |
+---+-------------+-------+
|1  |spark        |started|
|2  |delta        |started|
|3  |unity_catalog|started|
+---+-------------+-------+



## Parse PDFs And Chunk Text

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    return '\n'.join(page.extract_text() or '' for page in reader.pages)

documents = []
for pdf_path in sorted(PDF_DIR.glob('*.pdf')):
    text = read_pdf(pdf_path).strip()
    if text:
        documents.append({'source': pdf_path.name, 'text': text})

if not documents:
    documents.append({
        'source': 'sample.txt',
        'text': 'Localbricks lets us practice Spark, Delta Lake, Unity Catalog, PDF chunking, and LLM tool use before Databricks is provisioned.'
    })

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = []
for doc in documents:
    for chunk_id, chunk in enumerate(splitter.split_text(doc['text'])):
        chunks.append({'source': doc['source'], 'chunk_id': chunk_id, 'text': chunk})

chunks[:3]

In [ ]:
pdf_chunks_path = WAREHOUSE / 'demo' / 'pdf_chunks'

spark.sql('DROP TABLE IF EXISTS demo.pdf_chunks')
shutil.rmtree(pdf_chunks_path, ignore_errors=True)

chunks_df = spark.createDataFrame(chunks)
chunks_df.write.format('delta').mode('overwrite').save(pdf_chunks_path.as_posix())

spark.sql(f'''
CREATE TABLE demo.pdf_chunks
USING delta
LOCATION '{pdf_chunks_path.as_posix()}'
''')

spark.sql('SELECT source, chunk_id, substring(text, 1, 120) AS preview FROM demo.pdf_chunks ORDER BY source, chunk_id').show(truncate=False)

## Register A Unity Catalog AI Tool

In [ ]:
from unitycatalog.client import ApiClient, Configuration
from unitycatalog.ai.core.client import UnitycatalogFunctionClient

config = Configuration(host=UC_API_URL)
api_client = ApiClient(configuration=config)
uc_client = UnitycatalogFunctionClient(api_client=api_client)

CATALOG = 'unity'
SCHEMA = 'demo'

try:
    uc_client.uc.create_schema(name=SCHEMA, catalog_name=CATALOG, comment='Training schema for localbricks')
except Exception as exc:
    print(f'Schema already exists or could not be created: {exc}')

In [ ]:
def count_words(text: str) -> int:
    """
    Count whitespace-separated words in a text string.

    Args:
        text: Input text to count.

    Returns:
        Number of whitespace-separated words in the input text.
    """
    return len(text.split())

function_name = f'{CATALOG}.{SCHEMA}.count_words'

try:
    uc_client.create_python_function(func=count_words, catalog=CATALOG, schema=SCHEMA, replace=True)
except TypeError:
    try:
        uc_client.delete_function(function_name)
    except Exception:
        pass
    uc_client.create_python_function(func=count_words, catalog=CATALOG, schema=SCHEMA)

result = uc_client.execute_function(function_name, {'text': 'Spark and Delta tables are local today.'})
print(f'{function_name} returned: {result.value}')

## Use The UC Tool With LangChain And OpenAI

In [ ]:
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

toolkit = UCFunctionToolkit(function_names=[function_name], client=uc_client)
tools = toolkit.tools

print(tools[0].name)
print(tools[0].invoke({'text': 'Unity Catalog functions can be exposed as LangChain tools.'}))

In [ ]:
if not os.getenv('OPENAI_API_KEY'):
    print('Skipping OpenAI call because OPENAI_API_KEY is not set in .env')
else:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage

    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke([
        HumanMessage(content='Use the available tool to count the words in: local notebooks make Databricks practice concrete')
    ])

    print(response)